# minlpkit クイックスタート

MINLP(混合整数非線形計画)を PySCIPOpt(SCIP)で解くとき、**なぜ遅いか**を `minlpkit` で
観測し、**どう直すか**を試すチュートリアル。流れは:

1. 小さなMINLPをその場で定義する(整数×連続の積を含むバッチ生産計画)
2. `mk.analyze` で観測量収集 + 診断
3. `mk.linearize_product` で改善(整数×連続の積の厳密線形化)を適用
4. `mk.compare_variants` で before/after を比較検証

この notebook はリポジトリ内(editable インストール済みの `.venv`)でそのまま
`import minlpkit` が通る。外部で試す場合は下のインストールセルを参照。

In [1]:
# インストール(このリポジトリ内では uv sync 済みの環境で実行されるため不要)。
# 単独の環境(Google Colab等)で試す場合は次を実行する:
#
#   %pip install pyscipopt pandas
#   %pip install git+https://github.com/ctenopoma/minlpkit  # 公開後に有効(現在はプレースホルダ)
#
# コアの実行時依存は pyscipopt/pandas/numpy/scipy のみ(ライブモニタ/チューニングのextrasは不要)。


## 1. モデル定義: バッチ生産計画

ジョブ `j` ごとに整数のバッチ数 `n_j` と連続のバッチサイズ `s_j` を決め、
有効生産量 `n_j * s_j` が需要 `demand_j` を満たすようにしつつ、
段取り費用 `cost_n_j * n_j` とサイズ比例費用 `cost_s_j * s_j` の合計を最小化する。

`n_j * s_j`(整数×連続の双線形項)は SCIP がそのまま解くと McCormick 緩和になり、
分枝限定法で空間分枝(spatial branching)が必要になる。`linearize=True` にすると
`mk.linearize_product` で厳密線形化(McCormickギャップ0)した版に切り替わる。

In [2]:
import minlpkit as mk
from pyscipopt import Model, quicksum

JOBS = {
    "J1": dict(demand=12, cost_n=3.0, cost_s=1.0),
    "J2": dict(demand=18, cost_n=2.0, cost_s=1.5),
    "J3": dict(demand=9,  cost_n=4.0, cost_s=0.8),
}
N_MAX = 6
S_MAX = 10.0


def build_model(linearize: bool = False) -> Model:
    """バッチ生産計画。linearize=True で n_j*s_j を厳密線形化した版を返す。"""
    m = Model()
    m.hideOutput()
    n, s, ns = {}, {}, {}
    for j, d in JOBS.items():
        n[j] = m.addVar(vtype="I", lb=1, ub=N_MAX, name=f"n_{j}")
        s[j] = m.addVar(lb=0.0, ub=S_MAX, name=f"s_{j}")
        if linearize:
            ns[j] = mk.linearize_product(m, n[j], s[j], 1, N_MAX, 0.0, S_MAX, f"ns_{j}")
        else:
            ns[j] = n[j] * s[j]  # 双線形のまま(McCormick緩和)
        m.addCons(ns[j] >= d["demand"], name=f"demand_{j}")
    m.setObjective(
        quicksum(JOBS[j]["cost_n"] * n[j] + JOBS[j]["cost_s"] * s[j] for j in JOBS),
        "minimize",
    )
    m.data = dict(n=n, s=s)
    return m


print("baseline (bilinear):", build_model(False))
print("improved (linearized):", build_model(True))


baseline (bilinear): <pyscipopt.scip.Model object at 0x000001917EC1F130>
improved (linearized): <pyscipopt.scip.Model object at 0x000001917EC1F130>


## 2. 観測 + 診断 (`mk.analyze`)

`analyze` はモデルを実際に解いて双対境界の停滞・空間分枝の偏り・非線形制約の違反・
係数スケール・対称性などを収集し、診断ルールに照らして症状を重要度順に返す。

In [3]:
report = mk.analyze(lambda: build_model(False), name="baseline(bilinear)", time_limit=10)
print(report.summary())
report.metrics


[baseline(bilinear)] 検出症状 0件:


{'gap': 0.0015131339227919646,
 'spatial_share': 0.0,
 'n_stalls': 0,
 'nodes': 1,
 'bottleneck_type': 'demand',
 'bottleneck_rel_viol': 0.49493015597006484,
 'coef_ratio': 12.5,
 'bigm_count': 0,
 'residual_coef_ratio': 12.5,
 'residual_bigm_count': 0,
 'max_linking_groups': 1,
 'n_heavy_linking': 3,
 'n_sym_groups': 0,
 'largest_sym_group': 0,
 'sym_sound': False}

## 3. 改善を適用

診断の推薦どおり、整数×連続の積 `n_j * s_j` を `mk.linearize_product` で厳密線形化した
`improved` モデルを用意する(セル2の `build_model(linearize=True)`)。McCormick緩和の
ギャップが消えるため、ルート緩和での双対境界がタイトになるはず。

In [4]:
baseline = lambda: build_model(linearize=False)
improved = lambda: build_model(linearize=True)


## 4. before/after 検証 (`mk.compare_variants`)

ルート双対境界・時間制限内の最終gap・ノード数を1つの表にまとめて比較する。

In [5]:
df = mk.compare_variants({"baseline": baseline, "improved": improved}, time_limit=10)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]


,variant,root_dual,final_dual,final_gap,nodes
0,baseline,37.95,37.892663,0.001513,1
1,improved,37.95,37.950000,0.000000,1


## まとめ

- `mk.analyze` で症状を観測し、`recipe` に沿って `mk.linearize_product` / `mk.pwl_sos2` /
  `mk.benders` / `mk.column_generation` などの改善ヘルパーを適用する
- `mk.compare_variants` で改善前後をルート双対境界・gap・ノード数で定量比較する
- より大きな実モデル(バッチ反応器スケジューリング・Unit Commitment等)や、
  求解プロセスのライブ可視化(`minlpkit.live`)、SCIPパラメータの自動チューニング
  (`minlpkit.tune`)は [利用マニュアル](../docs/manual/index.md) と
  [成果ギャラリー](../docs/gallery.md) を参照。

> 技術ノート: この notebook は Google Colab でも実行できる(pyscipopt wheel に SCIP が
> ネイティブ同梱されているため)。JupyterLite などブラウザ内 Python 実行では SCIP の
> ネイティブバイナリを含められないため実行できない。